In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install cyvcf2 --quiet
!apt-get update
!apt-get install -y bcftools tabix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.7 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Pack

In [4]:
import pandas as pd
import numpy as np
import datetime
import os
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from cyvcf2 import VCF

In [7]:
path = "/content/drive/MyDrive/FYP_DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr21_sas_normalized.vcf.gz"


In [8]:
vcf = VCF(path)
multi_alt_count = 0
total = 0

for variant in vcf:
    total += 1
    if len(variant.ALT) > 1:
        multi_alt_count += 1
        print(f"Multi-allelic found: {variant.CHROM}:{variant.POS} {variant.REF} -> {variant.ALT}")

print(f"\nTotal variants checked: {total:,}")
print(f"Variants with >1 ALT allele: {multi_alt_count:,}")
if multi_alt_count == 0:
    print("✅ All variants are biallelic. No remaining multi-allelic sites.")
else:
    print("⚠️ Some multi-allelic sites remain.")


Total variants checked: 1,512,370
Variants with >1 ALT allele: 0
✅ All variants are biallelic. No remaining multi-allelic sites.


In [9]:
# --- Paths ---
dict_save_path = '/content/drive/MyDrive/FYP_DATA/processed_data/ClinVar/clinvar_lookup.pkl'

In [10]:
clinvar_dict = pd.read_pickle(dict_save_path)

print("Loaded ClinVar dictionary with:", len(clinvar_dict), "entries")

Loaded ClinVar dictionary with: 2214001 entries


In [11]:
# sanity: sample 5 keys
sample_keys = list(clinvar_dict.keys())[:5]
print("Sample clinvar keys:", sample_keys)

# quick check types
k = sample_keys[0]
assert isinstance(k[0], str) and isinstance(k[1], int) and isinstance(k[2], str) and isinstance(k[3], str)


Sample clinvar keys: [('chr1', 69134, 'A', 'G'), ('chr1', 69308, 'A', 'G'), ('chr1', 69314, 'T', 'G'), ('chr1', 69404, 'T', 'C'), ('chr1', 69423, 'G', 'A')]


In [13]:
output_csv = '/content/drive/MyDrive/FYP_DATA/processed_data/labeled_gnomad/gnomad_chr21_labeled.csv'


In [21]:
pd.DataFrame(columns=['CHROM','POS','REF','ALT','CLNSIG_LABEL']).to_csv(output_csv, index=False)


In [22]:
vcf = VCF(path)
batch = []
batch_size = 100_000
count = 0

for var in vcf:
    chrom = str(var.CHROM).strip()
    pos = int(var.POS)
    ref = str(var.REF).strip().upper()
    alt = str(var.ALT[0]).strip().upper()

    key = (chrom, pos, ref, alt)
    label = clinvar_dict.get(key, 'NA')

    batch.append((chrom, pos, ref, alt, label))
    count += 1

    if len(batch) >= batch_size:
        pd.DataFrame(batch).to_csv(output_csv, mode='a', index=False, header=False)
        print(f"Wrote {count:,} variants...")
        batch = []

# write leftover batch
if batch:
    pd.DataFrame(batch).to_csv(output_csv, mode='a', index=False, header=False)

print("🔥 Finished labeling chr21!")
print(f"Total variants processed: {count:,}")


Wrote 100,000 variants...
Wrote 200,000 variants...
Wrote 300,000 variants...
Wrote 400,000 variants...
Wrote 500,000 variants...
Wrote 600,000 variants...
Wrote 700,000 variants...
Wrote 800,000 variants...
Wrote 900,000 variants...
Wrote 1,000,000 variants...
Wrote 1,100,000 variants...
Wrote 1,200,000 variants...
Wrote 1,300,000 variants...
Wrote 1,400,000 variants...
Wrote 1,500,000 variants...
🔥 Finished labeling chr21!
Total variants processed: 1,512,370


In [23]:
df_new = pd.read_csv(output_csv)
df_new.head()

/tmp/ipython-input-3435271063.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_new = pd.read_csv(output_csv)


,CHROM,POS,REF,ALT,CLNSIG_LABEL
0,chr21,5030240,AC,A,NaN
1,chr21,5030446,C,G,NaN
2,chr21,5030495,C,G,NaN
3,chr21,5030521,G,A,NaN
4,chr21,5030526,G,A,NaN


In [25]:
df_new['CLNSIG_LABEL'].unique()

array([nan, 'Other', 'VUS', 'Benign', 'Likely_Benign', 'Pathogenic',
       'Conflicting_Classifications_of_Pathogenicity'], dtype=object)

In [28]:
df_new = df_new.dropna(subset=['CLNSIG_LABEL'])

In [29]:
len(df_new)

6259

In [31]:
df_new['CLNSIG_LABEL'].value_counts()

,count
CLNSIG_LABEL,
VUS,4440
Other,672
Likely_Benign,595
Benign,505
Pathogenic,46
Conflicting_Classifications_of_Pathogenicity,1


In [32]:
df_new.to_csv(output_csv, index=False)